In [1]:
import os
import json
import pandas as pd
import humanize
import re
import numpy as np
from scipy.spatial.transform import Rotation
import plotly.graph_objects as go

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
def split_vec3(s):
  vec3 = [float(x) for x in s.strip('()[]').split(',')]
  return vec3


def matrix_str_to_euler(rotation_str):
  """
  Парсит строку с матрицей поворота и возвращает углы Эйлера (в градусах).
  Формат строки: "[X: (x, y, z), Y: (x, y, z), Z: (x, y, z)]"
  
  Возвращает:
      tuple: (угол_X, угол_Y, угол_Z) в градусах
  """
  # Извлекаем числа из строки с помощью регулярного выражения
  numbers = re.findall(r"[-+]?\d*\.\d+|\d+", rotation_str)
  if len(numbers) != 9:
      raise ValueError("Некорректный формат матрицы поворота")
  
  # Преобразуем строки в числа float
  matrix = np.array([float(x) for x in numbers]).reshape(3, 3)
  
  # Конвертируем матрицу поворота в углы Эйлера (XYZ порядок)
  rot = Rotation.from_matrix(matrix)
  euler_angles = rot.as_euler('xyz', degrees=True)
  
  return tuple(euler_angles)

In [4]:
data = []
folder = 'proj_v2'
for fn in os.listdir(folder):
  if fn.endswith('.json'):
    with open(f"{folder}/{fn}", "r") as f:
      raw = json.load(f)
      data.append(raw)

In [5]:
df_raw = pd.json_normalize(data)
# df_raw.dropna(axis=0, how='all', inplace=True)
df_raw[['ang_vel.x', 'ang_vel.y', 'ang_vel.z']] = df_raw['angular_velocity'].apply(lambda x: split_vec3(x)).to_list()
df_raw[['position.x', 'position.y', 'position.z']] = df_raw['position'].apply(lambda x: split_vec3(x)).to_list()
df_raw[['velocity.x', 'velocity.y', 'velocity.z']] = df_raw['velocity'].apply(lambda x: split_vec3(x)).to_list()
df_raw['velocity_length'] = (df_raw['velocity.x']**2 + df_raw['velocity.y']**2 + df_raw['velocity.z']**2)**0.5
df_raw['kinetic_energy'] = 0.5 * (df_raw['mass'] / 1000) * df_raw['velocity_length']**2
df_raw[['euler.x', 'euler.y', 'euler.z']] = df_raw['rotation'].apply(lambda x: matrix_str_to_euler(x)).to_list()
df_raw.drop(['angular_velocity', 'position', 'velocity'], axis=1, inplace=True)
df_raw.columns

Index(['ammo', 'ammo.name', 'caliber', 'core_caliber', 'core_hardness',
       'core_mass', 'cross_section', 'drag_coef', 'flight_time',
       'fragmentation_chance', 'fragmentation_count', 'fragments_max',
       'fragments_min', 'impact_count', 'kind', 'length',
       'length_diameter_ratio', 'magnus_effect_factor', 'mass',
       'penetration_count', 'ricochet_count', 'ricochet_max_angle',
       'ricochet_min_angle', 'rotation', 'spin_decay_rate', 'state', 'ttl',
       'uid', 'weapon', 'weapon.name', 'ang_vel.x', 'ang_vel.y', 'ang_vel.z',
       'position.x', 'position.y', 'position.z', 'velocity.x', 'velocity.y',
       'velocity.z', 'velocity_length', 'kinetic_energy', 'euler.x', 'euler.y',
       'euler.z'],
      dtype='object')

In [6]:
cols = ['weapon.name', 'ammo.name', 'uid', 'flight_time',
        'position.x', 'position.y', 'position.z',
        'velocity.x', 'velocity.y', 'velocity.z', 'velocity_length',
        'ang_vel.x', 'ang_vel.y', 'ang_vel.z',
        'euler.x', 'euler.y', 'euler.z',
        'kinetic_energy'
      ]
repl = {
  'SCAR_L_CQC': 'SCAR-L CQC',
  'FN_EVOLYS': 'FN EVOLYS',
  'Glock17_HST': 'Glock 17',
  '9x19_HST_PlusP': '9x19 HST +P',
  '5.56x45_M855': '5.56x45 M855',
  '7.62x51_SLAP_T': '7.62x51 SLAP-T'
}
df = df_raw[cols].sort_values('flight_time').reset_index(drop=True)
df['weapon'] = df['weapon.name'].replace(repl, regex=False) + \
  ' (' + \
  df['ammo.name'].replace(repl, regex=False) + \
  ')'
df.drop(['weapon.name', 'ammo.name'], axis=1, inplace=True)
df[-2:]

,uid,flight_time,position.x,position.y,position.z,velocity.x,velocity.y,velocity.z,velocity_length,ang_vel.x,ang_vel.y,ang_vel.z,euler.x,euler.y,euler.z,kinetic_energy,weapon
240,proj_FnQQB5xr,0.0,-0.235469,2.150880,-87.71703,-2.239086,9.057234,-834.1066,834.158778,0.0,0.0,-32447.97,-0.684065,0.160568,25.853445,1426.432778,SCAR-L CQC (5.56x45 M855)
241,proj_FnQQB5xr,0.0,-2.591001,-0.657497,-965.20260,-0.788826,-8.917821,-293.8541,293.990445,0.0,0.0,-30695.72,-0.069137,0.699251,96.997398,177.182283,SCAR-L CQC (5.56x45 M855)


In [7]:
import plotly.graph_objects as go

fig = go.Figure()
markers = ['circle', 'square', 'diamond']
colors = ['red', 'blue', 'green']
cmin = df['velocity_length'].min()
cmax = df['velocity_length'].max()

for i, uid in enumerate(df['uid'].unique()[:3]):  # Ограничим 3 траекториями для наглядности
  df_subset = df.query(f"uid == '{uid}'")
  name = f"{i:3d}: {df_subset['weapon'].iloc[0]}"
  
  # Линия траектории
  fig.add_trace(go.Scatter3d(
    x=df_subset['position.x']+i,
    y=df_subset['position.z'],
    z=df_subset['position.y'],
    mode='lines',
    line=dict(width=2, color=colors[i]),
    name=name,
    showlegend=True,
  ))
  
  n = 10
  # Маркеры с текстовыми аннотациями (каждая 10-я точка)
  fig.add_trace(go.Scatter3d(
    x=df_subset['position.x'][::n]+i,
    y=df_subset['position.z'][::n],
    z=df_subset['position.y'][::n],
    mode='markers+text',  # Режим маркеров + текст
    marker=dict(
      size=4,
      symbol=markers[i],
      color=df_subset['velocity_length'][::n],
      colorscale='jet',
      opacity=0.8,
      cmin=cmin,  # Общие границы
      cmax=cmax,
      showscale=True if i == 0 else False
    ),
    text=[
      f"{humanize.metric(e['velocity_length'])}m/s {humanize.metric(e['kinetic_energy'])}J {e['flight_time']:.2f}s"
      for _, e in
      df_subset[['velocity_length', 'kinetic_energy', 'flight_time']][::n].iterrows()
    ],  # Текст аннотации
    textposition='top center',  # Позиция текста относительно точки
    textfont=dict(size=8, color=colors[i]),
    name='',
    showlegend=False  # Скрываем из легенды
  ))

# Настройка макета
fig.update_layout(
  width=1400,
  height=600,
  scene=dict(
    xaxis_title='X (m)',
    yaxis_title='Z (m)',
    zaxis_title='Y (m)',
    camera=dict(
      eye=dict(x=2, y=0, z=0),  # Камера смотрит вдоль оси X
      up=dict(x=0, y=0, z=1),   # Вертикаль — ось Z
      center=dict(x=0, y=0, z=0)  # Центр координат
    ),
    aspectmode='manual',
    aspectratio=dict(
      x=1.0,
      y=3.0,
      z=0.5
    ),
  ),
  legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01
  ),
  margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [8]:
df.groupby('uid').last()

,flight_time,position.x,position.y,position.z,velocity.x,velocity.y,velocity.z,velocity_length,ang_vel.x,ang_vel.y,ang_vel.z,euler.x,euler.y,euler.z,kinetic_energy,weapon
uid,,,,,,,,,,,,,,,,
proj_2GuRWFBB,0.0,-2.326464,1.850767,-866.6570,-0.891211,-6.785512,-331.9948,332.065332,0.0,0.0,-30988.79,-0.158536,0.684534,89.603551,226.048139,SCAR-L CQC (5.56x45 M855)
proj_FnQQB5xr,0.0,-2.591001,-0.657497,-965.2026,-0.788826,-8.917821,-293.8541,293.990445,0.0,0.0,-30695.72,-0.069137,0.699251,96.997398,177.182283,SCAR-L CQC (5.56x45 M855)


In [9]:
df_raw['rotation'][0]

'[X: (-0.898988, 0.437966, -0.002684), Y: (-0.437965, -0.898912, 0.011966), Z: (0.002828, 0.011933, 0.999925)]'